In [54]:
from copy import copy
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from scipy.optimize import bisect, brentq
from scipy.stats import norm

In [55]:
class CallOption:

    def __init__(self, K, T, price=None):
        self.K = K
        self.T = T
        self.price = price

In [56]:
class GBMdynamics:

    def __init__(self, X, r, rGrow, sigma=None):
        self.X = X
        self.r = r
        self.rGrow = rGrow
        self.sigma = sigma

    def update_sigma(self, sigma):
        self.sigma = sigma
        return self

In [57]:
class AnalyticEngine:

    def __init__(self):
        pass

    def BSpriceCall(self, dynamics, contract):
        # Ignores contract.price if given, because this function calculates price based on the dynamics.
        # Returns time-0 price.

        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        d2 = d1-std
        return np.exp(-dynamics.r*contract.T)*(F*norm.cdf(d1)-contract.K*norm.cdf(d2))

    def BSvega(self, dynamics, contract):
        # Returns time-$0$ vega

        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        return np.exp(-dynamics.r*contract.T)*F*norm.pdf(d1)*np.sqrt(contract.T)

    def IV(self, dynamics, contract):
        # ignores dynamics.sigma, because this function solves for sigma.
        # Returns time-$0$ implied volatility

        if contract.price is None:
            raise ValueError('Contract price must be given')

        df = np.exp(-dynamics.r*contract.T)  #discount factor
        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        lowerbound = np.max([0, df*(F - contract.K)])
        C = contract.price
        if C < lowerbound:
            return np.nan
        if C == lowerbound:
            return 0
        if C >= F*df:
            return np.nan

        dynamics_try = copy(dynamics)
        # We "try" values of sigma until we find sigma that generates price C

        # First find lower and upper bounds
        sigma_try = 0.2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) > C:
            sigma_try /= 2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) < C:
            sigma_try *= 2
        hi = sigma_try
        lo = hi/2
        # We have calculated "lo" and "hi" which bound the implied volatility from below and above.
        # In other words, the implied volatility is somewhere in the interval [lo,hi].
        # Then, to calculate the implied volatility within that interval,
        # for purposes of this homework, you may either (A) write your own bisection algorithm,
        # or (B) use scipy.optimize.bisect or (C) use scipy.optimize.brentq
        # You will need to provide lo and hi to those solvers.
        # There are other solvers that do not require you to bound the solution
        # from below and above (for instance, scipy.optimize.fsolve is a useful solver).
        # However, if you are able to bound the solution (of a single-variable problem),
        # then bisection or Brent will be more reliable.

        impliedVolatility = brentq(lambda sigma: self.BSpriceCall(dynamics_try.update_sigma(sigma), contract) - C, lo, hi)

        return impliedVolatility

## Problem 1

In [58]:
options = pd.read_csv('hw1-rut.csv')   #You may need to provide a path to the location of this file
options.set_index('Strike', drop=False, inplace=True)

T = 25/365
r = 0.0432

In [59]:
# Calculate the forward price.

min_K = float('inf')
for x in range(len(options)):
    if abs((options["CallAsk"].iloc[x] + options["CallBid"].iloc[x])/2 - (options["PutAsk"].iloc[x] + options["PutBid"].iloc[x])/2) < min_K:
        min_K = abs((options["CallAsk"].iloc[x] + options["CallBid"].iloc[x])/2 - (options["PutAsk"].iloc[x] + options["PutBid"].iloc[x])/2)
        min_K_index = x

F = options["Strike"].iloc[min_K_index] + np.exp(r*T) * ((options["CallAsk"].iloc[min_K_index] + options["CallBid"].iloc[min_K_index])/2 - (options["PutAsk"].iloc[min_K_index] + options["PutBid"].iloc[min_K_index])/2)
F

np.float64(2265.4011853143948)

In [60]:
options['ImpliedCallMid'] = (options["PutAsk"] + options["PutBid"])/2 + np.exp(-r*T) * (F - options['Strike']) # Fill this in with an "implied" call price obtained from the (midpoint) put price at each strike
options

,Strike,CallBid,CallAsk,PutBid,PutAsk,ImpliedCallMid
Strike,,,,,,
1750,1750,509.10,521.60,0.95,1.45,515.078417
1760,1760,499.20,511.70,1.00,1.50,505.157962
1770,1770,489.20,501.80,1.05,1.55,495.237507
1780,1780,479.30,491.90,1.10,1.65,485.342053
1790,1790,469.40,482.00,1.20,1.70,475.446598
...,...,...,...,...,...,...
2590,2590,0.25,0.85,317.20,330.80,0.360223
2600,2600,0.20,0.80,327.20,340.70,0.339768
2610,2610,0.15,0.75,337.10,350.70,0.319313


In [61]:
options['CallOrImpliedCall'] = options.apply(lambda row: (row["CallAsk"] + row["CallBid"])/2 if (row["Strike"] > F) else row['ImpliedCallMid'], axis=1)

# At strikes where the call is OTM, assign this to be the (midpoint) call price.
# At strikes where the put is OTM, assign this to be the implied (midpoint) call price, obtained from the (midpoint) put price

options

,Strike,CallBid,CallAsk,PutBid,PutAsk,ImpliedCallMid,CallOrImpliedCall
Strike,,,,,,,
1750,1750,509.10,521.60,0.95,1.45,515.078417,515.078417
1760,1760,499.20,511.70,1.00,1.50,505.157962,505.157962
1770,1770,489.20,501.80,1.05,1.55,495.237507,495.237507
1780,1780,479.30,491.90,1.10,1.65,485.342053,485.342053
1790,1790,469.40,482.00,1.20,1.70,475.446598,475.446598
...,...,...,...,...,...,...,...
2590,2590,0.25,0.85,317.20,330.80,0.360223,0.550000
2600,2600,0.20,0.80,327.20,340.70,0.339768,0.500000
2610,2610,0.15,0.75,337.10,350.70,0.319313,0.450000


In [62]:
hw1analytic = AnalyticEngine()
dynamics = GBMdynamics(X=F, r=r, rGrow=0, sigma=None)

In [63]:
options['IV'] = options.apply(lambda row: hw1analytic.IV(dynamics,  CallOption(K=row.Strike, T=T, price=row.CallOrImpliedCall)), axis=1)

In [64]:

fig = go.Figure()

fig.add_trace(go.Scatter(x=options.Strike, y=options.IV, mode='markers', name='Implied Volatility'))

fig.update_layout(title='Implied Volatility vs. Strike',
                  xaxis_title='Strike Price',
                  yaxis_title='Implied Volatility')
fig.show()


## Problem 2

In [65]:

alpha0, alpha1, alpha2, alpha3, alpha4 = np.polyfit(np.log(options.Strike / F), options.IV, 4)  # Fill this in by using a function such as numpy.polyfit to solve for array of coefficients (alpha0, ... , alpha4)

k = np.log(options.Strike / F)

coefficients = [alpha0, alpha1, alpha2, alpha3, alpha4]  # Fill this in by using a function such as numpy.polyfit to solve for array of coefficients (alpha0, ... , alpha4)
polynomial = np.poly1d(coefficients)


In [66]:
k_grid = np.linspace(k.min(), k.max(), 100)
y_fit = polynomial(k_grid)

fig = go.Figure()

fig.add_trace(go.Scatter(x=k, y=options.IV, mode='markers', name='Observed midpoint implied volatility'))
fig.add_trace(go.Scatter(x=k_grid, y=y_fit, mode='lines', name='Fitted polynomial parameterization'))

fig.update_layout(title='Implied Volatility vs log-moneyness',
                  xaxis_title='log-moneyness',
                  yaxis_title='Implied Volatility')

fig.show()

In [67]:
overpricedStrike = options.Strike.iloc[(options.IV - polynomial(k)).argmax()]
overpricedStrike

np.int64(2570)

In [68]:
# The vega calculation for each strike should use the fitted polynomial implied vol for that strike
options['vega'] = options.apply(lambda row: hw1analytic.BSvega(GBMdynamics(F, r, 0, polynomial(np.log(row.Strike / F)) ), CallOption(K=row.Strike, T=T)), axis=1)

In [69]:
options['skewsensitivity'] = options["vega"] * k

In [70]:
import plotly.express as px

fig = px.line(options, x="Strike", y="skewsensitivity", title='Sensitivity to Skew Slope, vs. Strike')
fig.show()

In [71]:
whichContractGainsMostFromSlopeIncrease = options.Strike.iloc[options['skewsensitivity'].argmax()]
whichContractGainsMostFromSlopeIncrease

np.int64(2385)

In [72]:
whichContractLosesMostFromSlopeIncrease = options.Strike.iloc[options['skewsensitivity'].argmin()]
whichContractLosesMostFromSlopeIncrease

np.int64(2100)